# **Generating Embeddings**

## Configuration & Environment Setup

In [2]:
from pathlib import Path
import os
import sys
import json
import hashlib
import platform
import importlib.util

print("Basic libraries imported successfully.")

Basic libraries imported successfully.


In [3]:
from pathlib import Path

# ------------------------------------------------------------
# A. PROJECT PATHS
# ------------------------------------------------------------

PROJECT_ROOT = Path("/home/muhamed/repo/CTDDG")

REWORKED_ROOT = PROJECT_ROOT / "REWORKED_CODE"

PREPROCESSING_ROOT = (
    REWORKED_ROOT / "PREPROCESSING_OUTPUT"
)


# ------------------------------------------------------------
# B. CLEANED BINDINGDB TEST DATASETS
# ------------------------------------------------------------

BINDINGDB_CLEAN_ROOT = (
    PREPROCESSING_ROOT / "bindingdb" / "test"
)

TEST_FOLD_1002_103 = (
    BINDINGDB_CLEAN_ROOT
    / "test_4_org_1002_103"
    / "pairs_clean.csv.gz"
)

TEST_FOLD_1042_104 = (
    BINDINGDB_CLEAN_ROOT
    / "test_4_org_1042_104"
    / "pairs_clean.csv.gz"
)

TEST_FOLDS = {
    "1002_103": TEST_FOLD_1002_103,
    "1042_104": TEST_FOLD_1042_104,
}

EXPECTED_PROTEIN_COUNTS = {
    "1002_103": 103,
    "1042_104": 104,
}


# ------------------------------------------------------------
# C. ORIGINAL FASTA FILES — ID VERIFICATION ONLY
# ------------------------------------------------------------

ORIGINAL_TEST_ROOT = (
    PROJECT_ROOT / "datasets/data/bindingdb/test_dataset"
)

ORIGINAL_FOLD_DIRS = {
    "1002_103": (
        ORIGINAL_TEST_ROOT / "test_4_org_1002_103"
    ),
    "1042_104": (
        ORIGINAL_TEST_ROOT / "test_4_org_1042_104"
    ),
}


# ------------------------------------------------------------
# D. EMBEDDING MODEL CONFIGURATION
# ------------------------------------------------------------

# Encoder selection is pending checkpoint verification.
# Possible encoders:
# - ProSE / Bepler: 6165 dimensions
# - ProtTransBertBFD: 1024 dimensions

EMBEDDING_BACKEND = "PENDING_VERIFICATION"

PROSE_ROOT = PROJECT_ROOT / "prose"

PROSE_MODEL_PATH = (
    PROSE_ROOT / "saved_models/prose_mt_3x1024.sav"
)


# ------------------------------------------------------------
# E. OUTPUT DIRECTORIES
# ------------------------------------------------------------

EVALUATION_ROOT = (
    REWORKED_ROOT / "EVALUATION_OUTPUT"
)

EMBEDDINGS_OUTPUT_ROOT = (
    EVALUATION_ROOT / "embeddings"
)

EMBEDDING_OUTPUT_DIRS = {
    fold: EMBEDDINGS_OUTPUT_ROOT / fold
    for fold in TEST_FOLDS
}


# ------------------------------------------------------------
# F. EXECUTION SETTINGS
# ------------------------------------------------------------

RANDOM_SEED = 42

SMOKE_TEST_PROTEINS = 2

EMBEDDING_BATCH_SIZE = 1

RESUME_EXISTING = True

ENABLE_GPU_EMBEDDING = False


print("Configuration loaded successfully.")
print("Test folds:", list(TEST_FOLDS))

Configuration loaded successfully.
Test folds: ['1002_103', '1042_104']


In [4]:
print("=" * 65)
print("PYTHON ENVIRONMENT")
print("=" * 65)

print("Python version :", sys.version.split()[0])
print("Executable     :", sys.executable)
print("Operating system:", platform.system())
print("Hostname       :", platform.node())

print("\n" + "=" * 65)
print("DEPENDENCY AVAILABILITY")
print("=" * 65)

required_packages = [
    "numpy",
    "pandas",
    "torch",
    "h5py",
    "rdkit",
]

optional_packages = [
    "bio_embeddings",
    "transformers",
    "tqdm",
]

def check_package(package_name):
    try:
        return importlib.util.find_spec(package_name) is not None
    except (ImportError, ModuleNotFoundError, ValueError):
        return False

for package in required_packages:
    available = check_package(package)
    print(f"{package:<20} {'OK' if available else 'MISSING'}")

print("\nOptional packages:")

for package in optional_packages:
    available = check_package(package)
    print(f"{package:<20} {'OK' if available else 'NOT INSTALLED'}")


print("\n" + "=" * 65)
print("SLURM ENVIRONMENT")
print("=" * 65)

job_id = os.environ.get("SLURM_JOB_ID")

if job_id:
    print("Slurm job ID:", job_id)
    print("Allocated node:", os.environ.get("SLURMD_NODENAME", "Unknown"))
else:
    print("No Slurm job allocation detected.")
    print("Dataset inspection is safe here.")
    print("Run expensive embedding inference on an allocated compute node.")

PYTHON ENVIRONMENT
Python version : 3.11.16
Executable     : /home/muhamed/.conda/envs/ctddg_torch/bin/python
Operating system: Linux
Hostname       : node004

DEPENDENCY AVAILABILITY
numpy                OK
pandas               OK
torch                OK
h5py                 OK
rdkit                OK

Optional packages:
bio_embeddings       NOT INSTALLED
transformers         NOT INSTALLED
tqdm                 OK

SLURM ENVIRONMENT
Slurm job ID: 15454
Allocated node: node004


In [6]:

# ============================================================
# BLOCK 1.4 — CLEANED DATASET VERIFICATION
# ============================================================

import csv
import gzip

REQUIRED_COLUMNS = {
    "source_line",
    "protein_sequence",
    "original_smiles",
    "canonical_smiles",
    "num_atoms",
    "num_bonds",
    "fragmented",
}

print("=" * 65)
print("CLEANED BINDINGDB DATASET VERIFICATION")
print("=" * 65)

for fold_name, dataset_path in TEST_FOLDS.items():

    print(f"\nFOLD: {fold_name}")
    print("Path:", dataset_path)

    if not dataset_path.is_file():
        raise FileNotFoundError(
            f"Cleaned dataset not found: {dataset_path}"
        )

    print("File exists: YES")
    print(
        "Compressed size:",
        round(dataset_path.stat().st_size / 1024**2, 2),
        "MB"
    )

    with gzip.open(
        dataset_path,
        "rt",
        encoding="utf-8",
        newline=""
    ) as f:

        reader = csv.DictReader(f)

        columns = set(reader.fieldnames or [])

        missing = REQUIRED_COLUMNS - columns

        if missing:
            raise ValueError(
                f"{fold_name}: Missing columns: {sorted(missing)}"
            )

        print("Required columns: VERIFIED")
        print("Columns:", reader.fieldnames)

        print("\nFirst 3 records:")

        for index in range(3):

            record = next(reader, None)

            if record is None:
                break

            sequence = record["protein_sequence"].strip()
            smiles = record["canonical_smiles"].strip()

            print(f"\nRecord {index + 1}")
            print("Source line:", record["source_line"])
            print("Protein length:", len(sequence))
            print("Sequence preview:", sequence[:70])
            print("Canonical SMILES:", smiles[:100])

    print("\n" + "-" * 65)

print("\nBLOCK 1 DATASET VERIFICATION COMPLETE")
print("No dataset files were modified.")


CLEANED BINDINGDB DATASET VERIFICATION

FOLD: 1002_103
Path: /home/muhamed/repo/CTDDG/REWORKED_CODE/PREPROCESSING_OUTPUT/bindingdb/test/test_4_org_1002_103/pairs_clean.csv.gz
File exists: YES
Compressed size: 1.06 MB
Required columns: VERIFIED
Columns: ['source_line', 'protein_sequence', 'original_smiles', 'canonical_smiles', 'num_atoms', 'num_bonds', 'fragmented']

First 3 records:

Record 1
Source line: 1
Protein length: 1390
Sequence preview: MKAPAVLAPGILVLLFTLVQRSNGECKEALAKSEMNVNMKYQLPNFTAETPIQNVILHEHHIFLGATNYI
Canonical SMILES: C/C(=N\OCCO)c1ccc2nnc([C@@H](C)c3cc4cnn(C)c4cc3C)n2n1

Record 2
Source line: 2
Protein length: 1390
Sequence preview: MKAPAVLAPGILVLLFTLVQRSNGECKEALAKSEMNVNMKYQLPNFTAETPIQNVILHEHHIFLGATNYI
Canonical SMILES: C/C(=N\OCCO)c1ccc2nnc([C@@H](C)c3c(F)cc4ncccc4c3F)n2n1

Record 3
Source line: 3
Protein length: 1390
Sequence preview: MKAPAVLAPGILVLLFTLVQRSNGECKEALAKSEMNVNMKYQLPNFTAETPIQNVILHEHHIFLGATNYI
Canonical SMILES: C/C(=N\OCC(O)CO)c1ccc2nnc(Cc3ccc4ncccc4c3)n2n1

## Discover, Extract & Validate Test Proteins

In [7]:
import csv
import gzip
import re
import hashlib
from pathlib import Path
from collections import Counter, defaultdict

import pandas as pd
from IPython.display import display

# ------------------------------------------------------------
# VERIFY CONFIGURATION FROM BLOCK 1
# ------------------------------------------------------------

assert "TEST_FOLDS" in globals(), (
    "Run Block 1.2 before Block 2."
)

assert "ORIGINAL_FOLD_DIRS" in globals(), (
    "Original FASTA paths are not configured."
)

for fold_name, file_path in TEST_FOLDS.items():
    assert file_path.is_file(), (
        f"Missing cleaned dataset: {file_path}"
    )

print("=" * 65)
print("BLOCK 2 — PROTEIN DATASET INSPECTION")
print("=" * 65)

print("\nCleaned test datasets:")

for fold_name, file_path in TEST_FOLDS.items():
    print(f"  {fold_name}: {file_path}")

print("\nAll input files found.")


BLOCK 2 — PROTEIN DATASET INSPECTION

Cleaned test datasets:
  1002_103: /home/muhamed/repo/CTDDG/REWORKED_CODE/PREPROCESSING_OUTPUT/bindingdb/test/test_4_org_1002_103/pairs_clean.csv.gz
  1042_104: /home/muhamed/repo/CTDDG/REWORKED_CODE/PREPROCESSING_OUTPUT/bindingdb/test/test_4_org_1042_104/pairs_clean.csv.gz

All input files found.


In [8]:
# ------------------------------------------------------------
# A. SEQUENCE HASH FUNCTION
# ------------------------------------------------------------

def sequence_sha256(sequence):
    return hashlib.sha256(
        sequence.encode("utf-8")
    ).hexdigest()


# ------------------------------------------------------------
# B. READ AND DEDUPLICATE PROTEINS
# ------------------------------------------------------------

def extract_unique_proteins(file_path):

    # Dictionary preserves order of first occurrence.
    proteins = {}

    total_pairs = 0
    previous_source_line = 0

    with gzip.open(
        file_path,
        "rt",
        encoding="utf-8",
        newline=""
    ) as f:

        reader = csv.DictReader(f)

        required = {
            "source_line",
            "protein_sequence",
            "canonical_smiles",
        }

        missing = required - set(reader.fieldnames or [])

        if missing:
            raise ValueError(
                f"Missing required columns: {sorted(missing)}"
            )

        for row in reader:

            total_pairs += 1

            sequence = row["protein_sequence"].strip()
            smiles = row["canonical_smiles"].strip()
            source_line = int(row["source_line"])

            if not sequence or not smiles:
                raise ValueError(
                    f"Empty sequence or SMILES in "
                    f"{file_path}, record {total_pairs}"
                )

            if source_line <= previous_source_line:
                raise ValueError(
                    f"Non-increasing source_line at "
                    f"record {total_pairs}: {source_line}"
                )

            previous_source_line = source_line

            if sequence not in proteins:

                proteins[sequence] = {
                    "sequence": sequence,
                    "sequence_sha256": sequence_sha256(sequence),
                    "length": len(sequence),
                    "first_source_line": source_line,
                    "last_source_line": source_line,
                    "ligand_pair_count": 0,
                }

            proteins[sequence]["ligand_pair_count"] += 1
            proteins[sequence]["last_source_line"] = source_line

    return list(proteins.values()), total_pairs


# ------------------------------------------------------------
# C. PROCESS BOTH TEST FOLDS
# ------------------------------------------------------------

FOLD_PROTEINS = {}
FOLD_STATS = {}

for fold_name, file_path in TEST_FOLDS.items():

    proteins, total_pairs = extract_unique_proteins(
        file_path
    )

    FOLD_PROTEINS[fold_name] = proteins

    FOLD_STATS[fold_name] = {
        "total_pairs": total_pairs,
        "unique_proteins": len(proteins),
        "expected_proteins": EXPECTED_PROTEIN_COUNTS[fold_name],
        "min_length": min(
            (p["length"] for p in proteins), default=0
        ),
        "max_length": max(
            (p["length"] for p in proteins), default=0
        ),
    }

    print("\n" + "=" * 65)
    print(f"FOLD: {fold_name}")
    print("=" * 65)

    print("Total protein-ligand pairs:", total_pairs)
    print("Unique protein sequences:", len(proteins))
    print(
        "Nominal expected proteins:",
        EXPECTED_PROTEIN_COUNTS[fold_name]
    )

    print(
        "Minimum sequence length:",
        FOLD_STATS[fold_name]["min_length"]
    )

    print(
        "Maximum sequence length:",
        FOLD_STATS[fold_name]["max_length"]
    )

    print("\nFirst 5 unique proteins:")

    for index, protein in enumerate(proteins[:5], start=1):

        print(f"\n  Sequence #{index}")
        print("    Length:", protein["length"])
        print("    Ligand pairs:", protein["ligand_pair_count"])
        print("    First source line:", protein["first_source_line"])
        print("    SHA256:", protein["sequence_sha256"][:16])
        print("    Preview:", protein["sequence"][:60])


print("\n" + "=" * 65)
print("UNIQUE PROTEIN EXTRACTION COMPLETE")
print("=" * 65)



FOLD: 1002_103
Total protein-ligand pairs: 16015
Unique protein sequences: 103
Nominal expected proteins: 103
Minimum sequence length: 81
Maximum sequence length: 2027

First 5 unique proteins:

  Sequence #1
    Length: 1390
    Ligand pairs: 2215
    First source line: 1
    SHA256: cc24e8701a5717fd
    Preview: MKAPAVLAPGILVLLFTLVQRSNGECKEALAKSEMNVNMKYQLPNFTAETPIQNVILHEH

  Sequence #2
    Length: 1367
    Ligand pairs: 1135
    First source line: 5
    SHA256: e11a8ca0d0d551f3
    Preview: MKSGSGGGSPTSLWGLLFLSAALSLWPTSGEICGPGIDIRNDYQQLKRLENCTVIEGYLH

  Sequence #3
    Length: 1388
    Ligand pairs: 2
    First source line: 6
    SHA256: 3f2d5da6cb870c0b
    Preview: MSRPPPTGKMPGAPEAVSGDGAGASRQRKLEALIRDPRSPINVESLLDGLNPLVLDLDFP

  Sequence #4
    Length: 1354
    Ligand pairs: 447
    First source line: 8
    SHA256: ea604ed7205f6778
    Preview: MSTGDSFETRFEKMDNLLRDPKSEVNSDCLLDGLDALVYDLDFPALRKNKNIDNFLSRYK

  Sequence #5
    Length: 1363
    Ligand pairs: 1
    First source line: 32

In [9]:
# ------------------------------------------------------------
# A. DEFINE STANDARD AMINO ACIDS
# ------------------------------------------------------------

STANDARD_AA = set("ACDEFGHIKLMNPQRSTVWY")

# These can occur in real protein datasets.
# Their compatibility depends on the selected encoder.
SPECIAL_AA = set("BXZJUO")

sequence_quality_rows = []


# ------------------------------------------------------------
# B. INSPECT EACH PROTEIN
# ------------------------------------------------------------

for fold_name, proteins in FOLD_PROTEINS.items():

    for index, protein in enumerate(proteins, start=1):

        sequence = protein["sequence"]

        characters = set(sequence)

        nonstandard = sorted(
            characters - STANDARD_AA
        )

        unsupported = sorted(
            characters - STANDARD_AA - SPECIAL_AA
        )

        sequence_quality_rows.append({
            "fold": fold_name,
            "temporary_id": f"seq{index}",
            "sequence_length": len(sequence),
            "ligand_pairs": protein["ligand_pair_count"],
            "sha256_prefix": protein["sequence_sha256"][:16],
            "nonstandard_aa": "".join(nonstandard),
            "unsupported_characters": "".join(unsupported),
            "longer_than_1024": len(sequence) > 1024,
            "longer_than_2048": len(sequence) > 2048,
        })


sequence_quality_df = pd.DataFrame(sequence_quality_rows)


# ------------------------------------------------------------
# C. DISPLAY SUMMARY
# ------------------------------------------------------------

print("=" * 65)
print("PROTEIN SEQUENCE QUALITY SUMMARY")
print("=" * 65)

summary = sequence_quality_df.groupby("fold").agg(
    proteins=("temporary_id", "count"),
    min_length=("sequence_length", "min"),
    median_length=("sequence_length", "median"),
    max_length=("sequence_length", "max"),
    over_1024=("longer_than_1024", "sum"),
    over_2048=("longer_than_2048", "sum"),
    nonstandard_count=(
        "nonstandard_aa",
        lambda x: (x != "").sum()
    ),
    unsupported_count=(
        "unsupported_characters",
        lambda x: (x != "").sum()
    ),
)

display(summary)


# ------------------------------------------------------------
# D. INSPECT LONGEST SEQUENCES
# ------------------------------------------------------------

print("\n10 longest proteins:")

display(
    sequence_quality_df.sort_values(
        "sequence_length",
        ascending=False
    ).head(10)
)


# ------------------------------------------------------------
# E. FLAG UNUSUAL AMINO ACIDS
# ------------------------------------------------------------

unusual = sequence_quality_df[
    sequence_quality_df["nonstandard_aa"] != ""
]

print("\nProteins containing nonstandard amino acids:", len(unusual))

if len(unusual):
    display(unusual.head(15))

print("\nSequence quality audit complete.")
print("No sequences have been modified or truncated.")


PROTEIN SEQUENCE QUALITY SUMMARY


,proteins,min_length,median_length,max_length,over_1024,over_2048,nonstandard_count,unsupported_count
fold,,,,,,,,
1002_103,103,81,1390.0,2027,95,0,0,0
1042_104,99,142,447.0,1980,9,0,0,0



10 longest proteins:


,fold,temporary_id,sequence_length,ligand_pairs,sha256_prefix,nonstandard_aa,unsupported_characters,longer_than_1024,longer_than_2048
36,1002_103,seq37,2027,10,1a318e559e0811b7,,,True,False
76,1002_103,seq77,2022,1,d23797c07dbcdf6a,,,True,False
94,1002_103,seq95,2016,516,3b1719ba4947964c,,,True,False
96,1002_103,seq97,2005,516,448e29b0b2d00f07,,,True,False
61,1002_103,seq62,2005,16,9da0258498b7efa8,,,True,False
87,1002_103,seq88,2000,516,92527863dc4b011c,,,True,False
91,1002_103,seq92,1988,516,6cf2c19bfc54eef2,,,True,False
65,1002_103,seq66,1984,3,b0290ba993ae5980,,,True,False
100,1002_103,seq101,1980,519,48b7226dc34e609c,,,True,False
140,1042_104,seq38,1980,519,48b7226dc34e609c,,,True,False



Proteins containing nonstandard amino acids: 0

Sequence quality audit complete.
No sequences have been modified or truncated.


In [10]:

# ============================================================
# BLOCK 2.4 — ORIGINAL FASTA ID RECONCILIATION
# ============================================================

# ------------------------------------------------------------
# A. FASTA READER
# ------------------------------------------------------------

def read_single_fasta(file_path):

    sequences = []
    current = []

    with file_path.open("r", encoding="utf-8") as f:

        for line in f:

            line = line.strip()

            if not line:
                continue

            if line.startswith(">"):

                if current:
                    sequences.append("".join(current))
                    current = []

            else:
                current.append(line)

    if current:
        sequences.append("".join(current))

    if len(sequences) != 1:
        raise ValueError(
            f"Expected exactly one FASTA sequence in "
            f"{file_path}; found {len(sequences)}"
        )

    return sequences[0]


# ------------------------------------------------------------
# B. DISCOVER LEGACY FASTA FILES
# ------------------------------------------------------------

FASTA_PATTERN = re.compile(
    r"protein([0-9]+)\.(?:fasta|fa|faa)$",
    re.IGNORECASE
)

FASTA_RECONCILIATION = {}
FASTA_AUDIT = {}

for fold_name, folder in ORIGINAL_FOLD_DIRS.items():

    print("\n" + "=" * 65)
    print(f"FASTA RECONCILIATION: {fold_name}")
    print("=" * 65)

    print("Original directory:", folder)

    old_files = []

    if folder.is_dir():

        for path in folder.iterdir():

            match = FASTA_PATTERN.fullmatch(path.name)

            if path.is_file() and match:
                old_files.append(
                    (int(match.group(1)), path)
                )

    old_files.sort(key=lambda x: x[0])

    print("Original FASTA files found:", len(old_files))

    # --------------------------------------------------------
    # LOAD LEGACY SEQUENCES
    # --------------------------------------------------------

    old_by_sequence = defaultdict(list)
    old_id_set = set()

    for protein_id, path in old_files:

        if protein_id in old_id_set:
            raise ValueError(
                f"Duplicate FASTA protein ID: {protein_id}"
            )

        old_id_set.add(protein_id)

        sequence = read_single_fasta(path)

        old_by_sequence[sequence].append({
            "protein_id": protein_id,
            "file": str(path),
        })

    # --------------------------------------------------------
    # MATCH CLEANED SEQUENCES
    # --------------------------------------------------------

    reconciled = []

    for index, protein in enumerate(
        FOLD_PROTEINS[fold_name], start=1
    ):

        candidates = old_by_sequence.get(
            protein["sequence"], []
        )

        if len(candidates) == 1:

            match_status = "EXACT_UNIQUE_MATCH"
            original_id = candidates[0]["protein_id"]

        elif len(candidates) > 1:

            match_status = "AMBIGUOUS_MULTIPLE_FASTA_IDS"
            original_id = None

        else:

            match_status = "NO_FASTA_MATCH"
            original_id = None

        reconciled.append({
            "fold": fold_name,
            "temporary_id": f"seq{index}",
            "original_protein_id": original_id,
            "match_status": match_status,
            "sequence_length": protein["length"],
            "ligand_pairs": protein["ligand_pair_count"],
            "first_source_line": protein["first_source_line"],
            "sequence_sha256": protein["sequence_sha256"],
            "matching_fasta_ids": [
                item["protein_id"]
                for item in candidates
            ],
        })

    result_df = pd.DataFrame(reconciled)

    FASTA_RECONCILIATION[fold_name] = result_df

    matched_sequences = set(
        protein["sequence"]
        for protein in FOLD_PROTEINS[fold_name]
    )

    old_only = [
        (seq, matches)
        for seq, matches in old_by_sequence.items()
        if seq not in matched_sequences
    ]

    FASTA_AUDIT[fold_name] = {
        "old_fasta_files": len(old_files),
        "cleaned_unique_proteins": len(result_df),
        "exact_unique_matches": int(
            (result_df["match_status"] == "EXACT_UNIQUE_MATCH").sum()
        ),
        "ambiguous_matches": int(
            (result_df["match_status"] ==
             "AMBIGUOUS_MULTIPLE_FASTA_IDS").sum()
        ),
        "unmatched_cleaned": int(
            (result_df["match_status"] == "NO_FASTA_MATCH").sum()
        ),
        "old_fasta_sequences_absent_from_cleaned": len(old_only),
    }

    # --------------------------------------------------------
    # DISPLAY RESULTS
    # --------------------------------------------------------

    print("\nMatch status counts:")

    display(
        result_df["match_status"]
        .value_counts()
        .rename_axis("status")
        .reset_index(name="count")
    )

    print("\nFirst 10 mappings:")

    display(
        result_df[
            [
                "temporary_id",
                "original_protein_id",
                "match_status",
                "sequence_length",
                "first_source_line",
            ]
        ].head(10)
    )

    print(
        "\nOriginal FASTA sequences absent from cleaned data:",
        len(old_only)
    )

    if not old_files:
        print(
            "NOTE: No matching original FASTA files found. "
            "Protein IDs remain unverified."
        )


print("\nFASTA reconciliation complete.")
print("No protein IDs were assigned by guessing.")



FASTA RECONCILIATION: 1002_103
Original directory: /home/muhamed/repo/CTDDG/datasets/data/bindingdb/test_dataset/test_4_org_1002_103
Original FASTA files found: 103

Match status counts:


,status,count
0,NO_FASTA_MATCH,102
1,AMBIGUOUS_MULTIPLE_FASTA_IDS,1



First 10 mappings:


,temporary_id,original_protein_id,match_status,sequence_length,first_source_line
0,seq1,None,NO_FASTA_MATCH,1390,1
1,seq2,None,NO_FASTA_MATCH,1367,5
2,seq3,None,NO_FASTA_MATCH,1388,6
3,seq4,None,NO_FASTA_MATCH,1354,8
4,seq5,None,NO_FASTA_MATCH,1363,32
5,seq6,None,NO_FASTA_MATCH,1159,33
6,seq7,None,NO_FASTA_MATCH,1363,80
7,seq8,None,NO_FASTA_MATCH,1070,88
8,seq9,None,NO_FASTA_MATCH,1634,90
9,seq10,None,NO_FASTA_MATCH,1388,106



Original FASTA sequences absent from cleaned data: 0

FASTA RECONCILIATION: 1042_104
Original directory: /home/muhamed/repo/CTDDG/datasets/data/bindingdb/test_dataset/test_4_org_1042_104
Original FASTA files found: 104

Match status counts:


,status,count
0,NO_FASTA_MATCH,99



First 10 mappings:


,temporary_id,original_protein_id,match_status,sequence_length,first_source_line
0,seq1,None,NO_FASTA_MATCH,683,1
1,seq2,None,NO_FASTA_MATCH,420,36
2,seq3,None,NO_FASTA_MATCH,427,127
3,seq4,None,NO_FASTA_MATCH,364,164
4,seq5,None,NO_FASTA_MATCH,323,171
5,seq6,None,NO_FASTA_MATCH,887,174
6,seq7,None,NO_FASTA_MATCH,366,230
7,seq8,None,NO_FASTA_MATCH,440,338
8,seq9,None,NO_FASTA_MATCH,395,344
9,seq10,None,NO_FASTA_MATCH,519,386



Original FASTA sequences absent from cleaned data: 1

FASTA reconciliation complete.
No protein IDs were assigned by guessing.


In [11]:

# ============================================================
# BLOCK 2.5 — FINAL DATASET AUDIT SUMMARY
# ============================================================

audit_rows = []

for fold_name in TEST_FOLDS:

    stats = FOLD_STATS[fold_name]
    fasta = FASTA_AUDIT[fold_name]

    audit_rows.append({
        "Fold": fold_name,
        "Cleaned pairs": stats["total_pairs"],
        "Unique proteins": stats["unique_proteins"],
        "Nominal expected": stats["expected_proteins"],
        "Count difference": (
            stats["unique_proteins"]
            - stats["expected_proteins"]
        ),
        "Original FASTA files": fasta["old_fasta_files"],
        "Exact ID matches": fasta["exact_unique_matches"],
        "Unmatched proteins": fasta["unmatched_cleaned"],
        "Ambiguous matches": fasta["ambiguous_matches"],
    })

audit_summary_df = pd.DataFrame(audit_rows)

print("=" * 65)
print("BLOCK 2 — FINAL AUDIT")
print("=" * 65)

display(audit_summary_df)

print("\n" + "=" * 65)
print("READINESS CHECK")
print("=" * 65)

for fold_name in TEST_FOLDS:

    row = audit_summary_df[
        audit_summary_df["Fold"] == fold_name
    ].iloc[0]

    count_ok = (
        row["Unique proteins"]
        == row["Nominal expected"]
    )

    mapping_ok = (
        row["Exact ID matches"]
        == row["Unique proteins"]
    )

    print(f"\nFold: {fold_name}")

    print(
        "Nominal protein count:",
        "MATCHES" if count_ok else "DIFFERS"
    )

    print(
        "Legacy protein ID mapping:",
        "FULLY VERIFIED" if mapping_ok
        else "REQUIRES REVIEW"
    )

print("\nBLOCK 2 COMPLETE.")
print("No embeddings generated.")
print("No source files modified.")


BLOCK 2 — FINAL AUDIT


,Fold,Cleaned pairs,Unique proteins,Nominal expected,Count difference,Original FASTA files,Exact ID matches,Unmatched proteins,Ambiguous matches
0,1002_103,16015,103,103,0,103,0,102,1
1,1042_104,11049,99,104,-5,104,0,99,0



READINESS CHECK

Fold: 1002_103
Nominal protein count: MATCHES
Legacy protein ID mapping: REQUIRES REVIEW

Fold: 1042_104
Nominal protein count: DIFFERS
Legacy protein ID mapping: REQUIRES REVIEW

BLOCK 2 COMPLETE.
No embeddings generated.
No source files modified.


## Recover Original Protein Identities

In [12]:

# ============================================================
# BLOCK 3.1 — ORIGINAL FASTA DUPLICATE AUDIT
# ============================================================

from collections import Counter
import hashlib
import re

print("=" * 65)
print("ORIGINAL FASTA DUPLICATE AUDIT")
print("=" * 65)

FASTA_DUPLICATE_AUDIT = {}

for fold_name, folder in ORIGINAL_FOLD_DIRS.items():

    print(f"\nFOLD: {fold_name}")

    files = sorted(
        [
            p for p in folder.iterdir()
            if p.is_file()
            and re.fullmatch(
                r"protein\d+\.(fasta|fa|faa)",
                p.name,
                re.IGNORECASE
            )
        ],
        key=lambda p: int(
            re.search(r"protein(\d+)", p.name).group(1)
        )
    ) if folder.is_dir() else []

    sequences = {
        p.name: read_single_fasta(p)
        for p in files
    }

    counts = Counter(sequences.values())

    FASTA_DUPLICATE_AUDIT[fold_name] = {
        "file_count": len(files),
        "unique_sequences": len(counts),
        "largest_duplicate_group": max(
            counts.values(), default=0
        ),
    }

    print("FASTA files:", len(files))
    print("Distinct sequences:", len(counts))
    print(
        "Largest identical-sequence group:",
        max(counts.values(), default=0)
    )

    print("\nFirst 5 files:")

    for filename, seq in list(sequences.items())[:5]:
        digest = hashlib.sha256(
            seq.encode()
        ).hexdigest()[:16]

        print(
            filename,
            "| length:", len(seq),
            "| SHA256:", digest
        )

    if len(files) > 1 and len(counts) == 1:
        print(
            "\nWARNING: Every original FASTA file "
            "contains the same protein sequence!"
        )

print("\nFASTA audit complete.")


ORIGINAL FASTA DUPLICATE AUDIT

FOLD: 1002_103
FASTA files: 103
Distinct sequences: 1
Largest identical-sequence group: 103

First 5 files:
protein1.fasta | length: 1560 | SHA256: 2e18349d95d39c5d
protein2.fasta | length: 1560 | SHA256: 2e18349d95d39c5d
protein3.fasta | length: 1560 | SHA256: 2e18349d95d39c5d
protein4.fasta | length: 1560 | SHA256: 2e18349d95d39c5d
protein5.fasta | length: 1560 | SHA256: 2e18349d95d39c5d


FOLD: 1042_104
FASTA files: 104
Distinct sequences: 1
Largest identical-sequence group: 104

First 5 files:
protein1.fasta | length: 1560 | SHA256: 2e18349d95d39c5d
protein2.fasta | length: 1560 | SHA256: 2e18349d95d39c5d
protein3.fasta | length: 1560 | SHA256: 2e18349d95d39c5d
protein4.fasta | length: 1560 | SHA256: 2e18349d95d39c5d
protein5.fasta | length: 1560 | SHA256: 2e18349d95d39c5d


FASTA audit complete.


In [13]:

# ============================================================
# BLOCK 3.2 — LOCATE RAW PROTEIN SOURCE FILES
# ============================================================

print("=" * 65)
print("RAW BINDINGDB PROTEIN SOURCE DISCOVERY")
print("=" * 65)

# The first location follows preprocessing_final.ipynb.
# The second location is the older project dataset root.

RAW_SOURCE_ROOTS = [
    REWORKED_ROOT / "DATA/bindingdb/test_dataset",
    PROJECT_ROOT / "datasets/data/bindingdb/test_dataset",
]

RAW_PROTEIN_FILES = {}

for fold_name in TEST_FOLDS:

    folder_name = f"test_4_org_{fold_name}"

    print(f"\nFOLD: {fold_name}")

    selected = None

    for root in RAW_SOURCE_ROOTS:

        folder = root / folder_name

        if not folder.is_dir():
            continue

        matches = sorted(
            p for p in folder.rglob("proteins_*_corrected")
            if p.is_file()
        )

        print("Searched:", folder)
        print("Matching source files:", len(matches))

        for path in matches:
            print(" ", path)

        if len(matches) > 1:
            raise ValueError(
                f"Ambiguous raw protein source for {fold_name}"
            )

        if len(matches) == 1 and selected is None:
            selected = matches[0]

    RAW_PROTEIN_FILES[fold_name] = selected

    if selected is None:
        print("WARNING: Raw corrected protein file not found.")
    else:
        print("Selected source:", selected)

print("\nSource discovery complete.")


RAW BINDINGDB PROTEIN SOURCE DISCOVERY

FOLD: 1002_103
Searched: /home/muhamed/repo/CTDDG/REWORKED_CODE/DATA/bindingdb/test_dataset/test_4_org_1002_103
Matching source files: 1
  /home/muhamed/repo/CTDDG/REWORKED_CODE/DATA/bindingdb/test_dataset/test_4_org_1002_103/proteins_test_4_org_corrected
Searched: /home/muhamed/repo/CTDDG/datasets/data/bindingdb/test_dataset/test_4_org_1002_103
Matching source files: 1
  /home/muhamed/repo/CTDDG/datasets/data/bindingdb/test_dataset/test_4_org_1002_103/proteins_test_4_org_corrected
Selected source: /home/muhamed/repo/CTDDG/REWORKED_CODE/DATA/bindingdb/test_dataset/test_4_org_1002_103/proteins_test_4_org_corrected

FOLD: 1042_104
Searched: /home/muhamed/repo/CTDDG/REWORKED_CODE/DATA/bindingdb/test_dataset/test_4_org_1042_104
Matching source files: 1
  /home/muhamed/repo/CTDDG/REWORKED_CODE/DATA/bindingdb/test_dataset/test_4_org_1042_104/proteins_test_4_org_corrected
Searched: /home/muhamed/repo/CTDDG/datasets/data/bindingdb/test_dataset/test_4_org

In [14]:

# ============================================================
# BLOCK 3.3 — RECONSTRUCT PROTEIN SOURCE MAPPING
# ============================================================

import csv
import gzip
import pandas as pd
from IPython.display import display

print("=" * 65)
print("PROTEIN SOURCE IDENTITY RECONSTRUCTION")
print("=" * 65)

PROTEIN_SOURCE_MAPPINGS = {}
MISSING_SOURCE_PROTEINS = {}

for fold_name, cleaned_path in TEST_FOLDS.items():

    raw_path = RAW_PROTEIN_FILES.get(fold_name)

    print(f"\n{'=' * 65}")
    print("FOLD:", fold_name)
    print("=" * 65)

    if raw_path is None:
        print("SKIPPED: Raw protein source is unavailable.")
        continue

    # --------------------------------------------------------
    # A. LOAD CLEANED SOURCE-LINE ASSOCIATIONS
    # --------------------------------------------------------

    cleaned_by_line = {}

    with gzip.open(
        cleaned_path, "rt",
        encoding="utf-8", newline=""
    ) as f:

        reader = csv.DictReader(f)

        for row in reader:

            line_number = int(row["source_line"])
            sequence = row["protein_sequence"].strip()

            if line_number in cleaned_by_line:
                raise ValueError(
                    f"Duplicate source line: {line_number}"
                )

            cleaned_by_line[line_number] = sequence

    # --------------------------------------------------------
    # B. READ ORIGINAL PROTEIN SOURCE IN ORDER
    # --------------------------------------------------------

    raw_first_occurrence = {}
    matched_records = 0
    raw_lines = 0

    with raw_path.open("r", encoding="utf-8") as f:

        for line_number, line in enumerate(f, start=1):

            raw_lines += 1

            # Same reconstruction as preprocessing_final.ipynb
            sequence = "".join(line.split())

            if sequence and sequence not in raw_first_occurrence:
                raw_first_occurrence[sequence] = {
                    "raw_order_id": len(raw_first_occurrence) + 1,
                    "first_source_line": line_number,
                }

            if line_number in cleaned_by_line:

                saved = cleaned_by_line[line_number]

                if sequence != saved:
                    raise ValueError(
                        f"{fold_name}: Protein mismatch "
                        f"at source line {line_number}"
                    )

                matched_records += 1

    if matched_records != len(cleaned_by_line):
        raise ValueError(
            f"{fold_name}: Only {matched_records}/"
            f"{len(cleaned_by_line)} cleaned rows "
            "matched original source lines."
        )

    # --------------------------------------------------------
    # C. IDENTIFY PROTEINS LOST DURING PREPROCESSING
    # --------------------------------------------------------

    cleaned_sequences = {
        item["sequence"]
        for item in FOLD_PROTEINS[fold_name]
    }

    missing_sequences = [
        seq for seq in raw_first_occurrence
        if seq not in cleaned_sequences
    ]

    missing_rows = []

    for seq in missing_sequences:

        source = raw_first_occurrence[seq]

        missing_rows.append({
            "raw_order_id": source["raw_order_id"],
            "first_source_line": source["first_source_line"],
            "sequence_length": len(seq),
            "sha256": hashlib.sha256(
                seq.encode()
            ).hexdigest(),
        })

    MISSING_SOURCE_PROTEINS[fold_name] = pd.DataFrame(
        missing_rows
    )

    # --------------------------------------------------------
    # D. MAP CLEANED PROTEINS TO THEIR RAW-SOURCE ORDER
    # --------------------------------------------------------

    mapping_rows = []

    for index, protein in enumerate(
        FOLD_PROTEINS[fold_name], start=1
    ):

        source = raw_first_occurrence.get(
            protein["sequence"]
        )

        if source is None:
            raise ValueError(
                f"Cleaned sequence {index} missing from raw source"
            )

        mapping_rows.append({
            "fold": fold_name,
            "cleaned_order_id": index,
            "raw_order_id": source["raw_order_id"],
            "first_source_line": source["first_source_line"],
            "first_retained_source_line":
                protein["first_source_line"],
            "sequence_length": protein["length"],
            "ligand_pairs": protein["ligand_pair_count"],
            "sequence_sha256": protein["sequence_sha256"],
        })

    mapping_df = pd.DataFrame(mapping_rows)

    PROTEIN_SOURCE_MAPPINGS[fold_name] = mapping_df

    # --------------------------------------------------------
    # E. DISPLAY RESULTS
    # --------------------------------------------------------

    print("Raw source lines:", raw_lines)
    print("Cleaned pairs:", len(cleaned_by_line))
    print("Source-line matches:", matched_records)
    print("Original unique proteins:", len(raw_first_occurrence))
    print("Cleaned unique proteins:", len(cleaned_sequences))
    print("Proteins completely filtered out:", len(missing_sequences))

    print("\nFirst 10 source mappings:")
    display(mapping_df.head(10))

    print("\nProteins missing after preprocessing:")

    if missing_rows:
        display(MISSING_SOURCE_PROTEINS[fold_name])
    else:
        print("None")

print("\nBLOCK 3 COMPLETE — Source identities inspected.")
print("No files written.")
print("No embeddings generated.")


PROTEIN SOURCE IDENTITY RECONSTRUCTION

FOLD: 1002_103
Raw source lines: 16015
Cleaned pairs: 16015
Source-line matches: 16015
Original unique proteins: 103
Cleaned unique proteins: 103
Proteins completely filtered out: 0

First 10 source mappings:


,fold,cleaned_order_id,raw_order_id,first_source_line,first_retained_source_line,sequence_length,ligand_pairs,sequence_sha256
0,1002_103,1,1,1,1,1390,2215,cc24e8701a5717fd1fa418174adb3b6cbaeae8a76f78b3...
1,1002_103,2,2,5,5,1367,1135,e11a8ca0d0d551f3fa704c31f4d0af57361c9d19d838a5...
2,1002_103,3,3,6,6,1388,2,3f2d5da6cb870c0bd39652a13b0eba159aaaed65d07ea6...
3,1002_103,4,4,8,8,1354,447,ea604ed7205f6778d46f9ba20f4fd1de8938ca8feb49d8...
4,1002_103,5,5,32,32,1363,1,10f416eb8a4e9ad3de99ed36ca82cdc480c35f60060459...
5,1002_103,6,6,33,33,1159,485,287332153da38b59cc1be9554cc3a29f14d3b9e2a33150...
6,1002_103,7,7,80,80,1363,309,e269bc198cd67f0c6b80d470b4d790fedd12e2902018ae...
7,1002_103,8,8,88,88,1070,944,549e89f067b7e11976787b867a566923f91938b9234ea1...
8,1002_103,9,9,90,90,1634,89,c4a41123ce734ae5702facde8accdc965f5d4b1217a3fb...
9,1002_103,10,10,106,106,1388,1381,2674aa9015e37da743be74d67bce80b3cb35df6161e918...



Proteins missing after preprocessing:
None

FOLD: 1042_104
Raw source lines: 11049
Cleaned pairs: 11049
Source-line matches: 11049
Original unique proteins: 99
Cleaned unique proteins: 99
Proteins completely filtered out: 0

First 10 source mappings:


,fold,cleaned_order_id,raw_order_id,first_source_line,first_retained_source_line,sequence_length,ligand_pairs,sequence_sha256
0,1042_104,1,1,1,1,683,201,6b12ced2223b354a36749591d4b03c12e25567a3e0dcc7...
1,1042_104,2,2,36,36,420,746,2ca842f3e7a01314833c340d495dc171c89dca6271a7cf...
2,1042_104,3,3,127,127,427,535,e21eccad4aa906413ff8b7fcb1a5b107b313731e405437...
3,1042_104,4,4,164,164,364,25,eb9d2c6479a483498c2279616607cbc096e3cb0668998a...
4,1042_104,5,5,171,171,323,273,7a787644dd8d6e14aeb85be1f0abaae0085dcfff35a891...
5,1042_104,6,6,174,174,887,519,e9dd462b4e9f801085cb0863bdb0fc8e2efeb894791693...
6,1042_104,7,7,230,230,366,837,73fe68d1fe5d48b8452464bee663af0788bcce6806ae4c...
7,1042_104,8,8,338,338,440,471,a9754aa0eed81bf0e9e2fd8ef8d35d6f3d58f467b794f3...
8,1042_104,9,9,344,344,395,2000,9800e73d5db87c78c7b5350307ddf15012927fada0dd1e...
9,1042_104,10,10,386,386,519,2,463fc69c83e6746aab8f198ea4f198bfc28dc86e726755...



Proteins missing after preprocessing:
None

BLOCK 3 COMPLETE — Source identities inspected.
No files written.
No embeddings generated.


## Create Verified Protein Manifest & FASTA Files

In [15]:

# ============================================================
# BLOCK 4.1 — BUILD VERIFIED PROTEIN MANIFEST
# ============================================================

import pandas as pd
from pathlib import Path

# ------------------------------------------------------------
# A. VERIFY PREVIOUS BLOCKS
# ------------------------------------------------------------

assert "FOLD_PROTEINS" in globals()
assert "PROTEIN_SOURCE_MAPPINGS" in globals()
assert "EMBEDDING_OUTPUT_DIRS" in globals()

ID_NAMESPACE = "cleaned_first_occurrence_v1"

PROTEIN_MANIFESTS = {}

# ------------------------------------------------------------
# B. CREATE ONE MANIFEST PER FOLD
# ------------------------------------------------------------

for fold_name, proteins in FOLD_PROTEINS.items():

    source_mapping = (
        PROTEIN_SOURCE_MAPPINGS[fold_name]
        .set_index("cleaned_order_id")
    )

    output_dir = EMBEDDING_OUTPUT_DIRS[fold_name]

    rows = []

    for index, protein in enumerate(proteins, start=1):

        source = source_mapping.loc[index]

        assert (
            protein["sequence_sha256"]
            == source["sequence_sha256"]
        ), f"{fold_name}: Sequence identity mismatch at {index}"

        assert (
            protein["length"]
            == int(source["sequence_length"])
        )

        protein_id = f"protein{index}"

        fasta_path = (
            output_dir / "sequences" / f"{protein_id}.fasta"
        )

        rows.append({
            "fold": fold_name,
            "protein_id": protein_id,
            "id_namespace": ID_NAMESPACE,
            "cleaned_order_id": index,
            "raw_order_id": int(source["raw_order_id"]),
            "first_source_line": int(
                source["first_source_line"]
            ),
            "first_retained_source_line": int(
                source["first_retained_source_line"]
            ),
            "sequence_length": protein["length"],
            "ligand_pair_count": protein["ligand_pair_count"],
            "sequence_sha256": protein["sequence_sha256"],
            "fasta_path": str(fasta_path),
        })

    manifest = pd.DataFrame(rows)

    PROTEIN_MANIFESTS[fold_name] = manifest

    print(f"\nFold: {fold_name}")
    print("Proteins:", len(manifest))
    print("ID namespace:", ID_NAMESPACE)

    display(manifest.head(5))

print("\nProtein manifests constructed in memory.")



Fold: 1002_103
Proteins: 103
ID namespace: cleaned_first_occurrence_v1


,fold,protein_id,id_namespace,cleaned_order_id,raw_order_id,first_source_line,first_retained_source_line,sequence_length,ligand_pair_count,sequence_sha256,fasta_path
0,1002_103,protein1,cleaned_first_occurrence_v1,1,1,1,1,1390,2215,cc24e8701a5717fd1fa418174adb3b6cbaeae8a76f78b3...,/home/muhamed/repo/CTDDG/REWORKED_CODE/EVALUAT...
1,1002_103,protein2,cleaned_first_occurrence_v1,2,2,5,5,1367,1135,e11a8ca0d0d551f3fa704c31f4d0af57361c9d19d838a5...,/home/muhamed/repo/CTDDG/REWORKED_CODE/EVALUAT...
2,1002_103,protein3,cleaned_first_occurrence_v1,3,3,6,6,1388,2,3f2d5da6cb870c0bd39652a13b0eba159aaaed65d07ea6...,/home/muhamed/repo/CTDDG/REWORKED_CODE/EVALUAT...
3,1002_103,protein4,cleaned_first_occurrence_v1,4,4,8,8,1354,447,ea604ed7205f6778d46f9ba20f4fd1de8938ca8feb49d8...,/home/muhamed/repo/CTDDG/REWORKED_CODE/EVALUAT...
4,1002_103,protein5,cleaned_first_occurrence_v1,5,5,32,32,1363,1,10f416eb8a4e9ad3de99ed36ca82cdc480c35f60060459...,/home/muhamed/repo/CTDDG/REWORKED_CODE/EVALUAT...



Fold: 1042_104
Proteins: 99
ID namespace: cleaned_first_occurrence_v1


,fold,protein_id,id_namespace,cleaned_order_id,raw_order_id,first_source_line,first_retained_source_line,sequence_length,ligand_pair_count,sequence_sha256,fasta_path
0,1042_104,protein1,cleaned_first_occurrence_v1,1,1,1,1,683,201,6b12ced2223b354a36749591d4b03c12e25567a3e0dcc7...,/home/muhamed/repo/CTDDG/REWORKED_CODE/EVALUAT...
1,1042_104,protein2,cleaned_first_occurrence_v1,2,2,36,36,420,746,2ca842f3e7a01314833c340d495dc171c89dca6271a7cf...,/home/muhamed/repo/CTDDG/REWORKED_CODE/EVALUAT...
2,1042_104,protein3,cleaned_first_occurrence_v1,3,3,127,127,427,535,e21eccad4aa906413ff8b7fcb1a5b107b313731e405437...,/home/muhamed/repo/CTDDG/REWORKED_CODE/EVALUAT...
3,1042_104,protein4,cleaned_first_occurrence_v1,4,4,164,164,364,25,eb9d2c6479a483498c2279616607cbc096e3cb0668998a...,/home/muhamed/repo/CTDDG/REWORKED_CODE/EVALUAT...
4,1042_104,protein5,cleaned_first_occurrence_v1,5,5,171,171,323,273,7a787644dd8d6e14aeb85be1f0abaae0085dcfff35a891...,/home/muhamed/repo/CTDDG/REWORKED_CODE/EVALUAT...



Protein manifests constructed in memory.


In [16]:

# ============================================================
# BLOCK 4.2 — MANIFEST INTEGRITY VERIFICATION
# ============================================================

STANDARD_AA = set("ACDEFGHIKLMNPQRSTVWY")

for fold_name, manifest in PROTEIN_MANIFESTS.items():

    proteins = FOLD_PROTEINS[fold_name]

    # Each sequence gets exactly one ID within its fold.
    assert len(manifest) == len(proteins)

    assert manifest["protein_id"].is_unique
    assert manifest["sequence_sha256"].is_unique

    expected_ids = [
        f"protein{i}"
        for i in range(1, len(proteins) + 1)
    ]

    assert manifest["protein_id"].tolist() == expected_ids

    # Confirm the manifest covers every retained pair.
    assert (
        manifest["ligand_pair_count"].sum()
        == FOLD_STATS[fold_name]["total_pairs"]
    )

    # Check every sequence before export.
    for protein in proteins:

        sequence = protein["sequence"]

        assert sequence
        assert set(sequence).issubset(STANDARD_AA)

        assert (
            sequence_sha256(sequence)
            == protein["sequence_sha256"]
        )

    print(
        f"PASS: {fold_name} | {len(proteins)} unique proteins "
        f"| {manifest['ligand_pair_count'].sum():,} pairs"
    )


# ------------------------------------------------------------
# CROSS-FOLD OVERLAP
# ------------------------------------------------------------

fold_names = list(PROTEIN_MANIFESTS)

left = set(
    PROTEIN_MANIFESTS[fold_names[0]]["sequence_sha256"]
)

right = set(
    PROTEIN_MANIFESTS[fold_names[1]]["sequence_sha256"]
)

shared_hashes = left & right

print("\nCross-fold overlap:")
print("Distinct sequences shared by both folds:", len(shared_hashes))

print("\nTotal fold-specific entries:", sum(
    len(m) for m in PROTEIN_MANIFESTS.values()
))

print("Unique sequences across both folds:", len(left | right))

print("\nBLOCK 4.2 PASSED.")


PASS: 1002_103 | 103 unique proteins | 16,015 pairs
PASS: 1042_104 | 99 unique proteins | 11,049 pairs

Cross-fold overlap:
Distinct sequences shared by both folds: 5

Total fold-specific entries: 202
Unique sequences across both folds: 197

BLOCK 4.2 PASSED.


In [17]:

# ============================================================
# BLOCK 4.3 — EXPORT VERIFIED PROTEIN FILES
# ============================================================

import json
import os
import hashlib
import tempfile

# ------------------------------------------------------------
# A. FORMAT FASTA SEQUENCES
# ------------------------------------------------------------

def make_fasta(protein_id, fold, sequence, sequence_hash):

    header = (
        f">{protein_id} "
        f"fold={fold} "
        f"sha256={sequence_hash}"
    )

    lines = [
        sequence[i:i + 80]
        for i in range(0, len(sequence), 80)
    ]

    return header + "\n" + "\n".join(lines) + "\n"


def hash_file(path):

    digest = hashlib.sha256()

    with Path(path).open("rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            digest.update(chunk)

    return digest.hexdigest()


# ------------------------------------------------------------
# B. PREPARE ALL OUTPUT FILES
# ------------------------------------------------------------

planned_files = {}

for fold_name, manifest in PROTEIN_MANIFESTS.items():

    output_dir = EMBEDDING_OUTPUT_DIRS[fold_name]
    proteins = FOLD_PROTEINS[fold_name]

    for row, protein in zip(
        manifest.itertuples(index=False),
        proteins
    ):

        fasta_content = make_fasta(
            row.protein_id,
            fold_name,
            protein["sequence"],
            protein["sequence_sha256"]
        )

        planned_files[Path(row.fasta_path)] = (
            fasta_content.encode("utf-8")
        )

    # Save sequence-to-ID mapping.
    csv_path = output_dir / "protein_manifest.csv"

    planned_files[csv_path] = (
        manifest.to_csv(index=False).encode("utf-8")
    )

    # Save provenance and source dataset checksums.
    metadata = {
        "fold": fold_name,
        "id_namespace": ID_NAMESPACE,
        "id_assignment": "first appearance in cleaned CSV",
        "protein_count": len(manifest),
        "nominal_expected_count":
            EXPECTED_PROTEIN_COUNTS[fold_name],
        "cleaned_pairs_file": str(TEST_FOLDS[fold_name]),
        "cleaned_pairs_sha256": hash_file(
            TEST_FOLDS[fold_name]
        ),
        "raw_protein_source": str(
            RAW_PROTEIN_FILES[fold_name]
        ),
        "raw_protein_source_sha256": hash_file(
            RAW_PROTEIN_FILES[fold_name]
        ),
        "legacy_fasta_ids_verified": False,
        "embedding_model": "PENDING_VERIFICATION",
        "embeddings_generated": False,
    }

    metadata_path = output_dir / "protein_metadata.json"

    planned_files[metadata_path] = (
        json.dumps(metadata, indent=2)
        .encode("utf-8")
    )


# ------------------------------------------------------------
# C. CHECK FOR EXISTING CONFLICTING FILES
# ------------------------------------------------------------

conflicts = []

for path, content in planned_files.items():

    if path.exists():

        if not path.is_file() or path.read_bytes() != content:
            conflicts.append(path)

if conflicts:
    raise FileExistsError(
        "Existing output files differ. Nothing was overwritten:\n"
        + "\n".join(str(p) for p in conflicts[:20])
    )


# ------------------------------------------------------------
# D. WRITE NEW FILES ATOMICALLY
# ------------------------------------------------------------

created = 0
reused = 0

for path, content in planned_files.items():

    if path.exists():
        reused += 1
        continue

    path.parent.mkdir(parents=True, exist_ok=True)

    temp_path = None

    try:
        with tempfile.NamedTemporaryFile(
            mode="wb",
            dir=path.parent,
            prefix=".writing_",
            delete=False
        ) as temporary:

            temporary.write(content)
            temp_path = Path(temporary.name)

        os.replace(temp_path, path)
        created += 1

    finally:
        if temp_path is not None and temp_path.exists():
            temp_path.unlink()


print("=" * 65)
print("PROTEIN FILE EXPORT")
print("=" * 65)

print("Files created:", created)
print("Matching existing files reused:", reused)

for fold_name, output_dir in EMBEDDING_OUTPUT_DIRS.items():

    print(f"\n{fold_name}:")
    print("  FASTA directory:", output_dir / "sequences")
    print("  Manifest:", output_dir / "protein_manifest.csv")
    print("  Metadata:", output_dir / "protein_metadata.json")

print("\nExport complete. Legacy source files were not modified.")


PROTEIN FILE EXPORT
Files created: 206
Matching existing files reused: 0

1002_103:
  FASTA directory: /home/muhamed/repo/CTDDG/REWORKED_CODE/EVALUATION_OUTPUT/embeddings/1002_103/sequences
  Manifest: /home/muhamed/repo/CTDDG/REWORKED_CODE/EVALUATION_OUTPUT/embeddings/1002_103/protein_manifest.csv
  Metadata: /home/muhamed/repo/CTDDG/REWORKED_CODE/EVALUATION_OUTPUT/embeddings/1002_103/protein_metadata.json

1042_104:
  FASTA directory: /home/muhamed/repo/CTDDG/REWORKED_CODE/EVALUATION_OUTPUT/embeddings/1042_104/sequences
  Manifest: /home/muhamed/repo/CTDDG/REWORKED_CODE/EVALUATION_OUTPUT/embeddings/1042_104/protein_manifest.csv
  Metadata: /home/muhamed/repo/CTDDG/REWORKED_CODE/EVALUATION_OUTPUT/embeddings/1042_104/protein_metadata.json

Export complete. Legacy source files were not modified.


In [18]:

# ============================================================
# BLOCK 4.4 — FINAL FASTA EXPORT VALIDATION
# ============================================================

total_verified = 0

for fold_name, output_dir in EMBEDDING_OUTPUT_DIRS.items():

    manifest_path = output_dir / "protein_manifest.csv"

    manifest = pd.read_csv(manifest_path)

    expected_proteins = FOLD_PROTEINS[fold_name]

    assert len(manifest) == len(expected_proteins)

    for row, protein in zip(
        manifest.itertuples(index=False),
        expected_proteins
    ):

        fasta_path = Path(row.fasta_path)

        assert fasta_path.is_file(), (
            f"Missing FASTA: {fasta_path}"
        )

        saved_sequence = read_single_fasta(fasta_path)

        # Exact sequence identity check.
        assert saved_sequence == protein["sequence"], (
            f"Sequence mismatch: {fasta_path}"
        )

        # Hash verification.
        saved_hash = sequence_sha256(saved_sequence)

        assert saved_hash == row.sequence_sha256, (
            f"Hash mismatch: {fasta_path}"
        )

        assert len(saved_sequence) == row.sequence_length

        total_verified += 1

    print(
        f"PASS: {fold_name} | "
        f"{len(manifest)} FASTA files verified"
    )

print("\n" + "=" * 65)
print("BLOCK 4 — FINAL SUMMARY")
print("=" * 65)

print("Total verified FASTA files:", total_verified)
print("Source-to-FASTA sequence identity: PASSED")
print("SHA-256 verification: PASSED")
print("Legacy files modified: NO")
print("Embeddings generated: NO")

print("\nBLOCK 4 COMPLETE.")


PASS: 1002_103 | 103 FASTA files verified
PASS: 1042_104 | 99 FASTA files verified

BLOCK 4 — FINAL SUMMARY
Total verified FASTA files: 202
Source-to-FASTA sequence identity: PASSED
SHA-256 verification: PASSED
Legacy files modified: NO
Embeddings generated: NO

BLOCK 4 COMPLETE.


## Embedding Model Compatibility Check

In [19]:

# ============================================================
# BLOCK 5.1 — ENCODER AND GPU AVAILABILITY
# ============================================================

import importlib.util
import torch

print("=" * 65)
print("EMBEDDING ENVIRONMENT")
print("=" * 65)

print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print(
        "GPU memory (GB):",
        round(
            torch.cuda.get_device_properties(0).total_memory
            / 1024**3, 2
        )
    )

print("\nAvailable Python packages:")

for name in ["bio_embeddings", "transformers", "prose"]:
    try:
        found = importlib.util.find_spec(name) is not None
    except (ImportError, ValueError, ModuleNotFoundError):
        found = False
    print(f"  {name}: {'INSTALLED' if found else 'NOT INSTALLED'}")

print("\nCandidate ProSE model locations:")

candidate_models = [
    PROJECT_ROOT / "prose/saved_models/prose_mt_3x1024.sav",
    PROJECT_ROOT / "saved_models/prose_mt_3x1024.sav",
    REWORKED_ROOT / "saved_models/prose_mt_3x1024.sav",
]

for path in candidate_models:
    print(
        "FOUND" if path.is_file() else "NOT FOUND",
        "|", path
    )

print("\nBLOCK 5.1 COMPLETE")


EMBEDDING ENVIRONMENT
PyTorch version: 2.4.1+cu121
CUDA available: True
GPU: NVIDIA L40
GPU memory (GB): 44.37

Available Python packages:
  bio_embeddings: NOT INSTALLED
  transformers: NOT INSTALLED
  prose: INSTALLED

Candidate ProSE model locations:
FOUND | /home/muhamed/repo/CTDDG/prose/saved_models/prose_mt_3x1024.sav
NOT FOUND | /home/muhamed/repo/CTDDG/saved_models/prose_mt_3x1024.sav
NOT FOUND | /home/muhamed/repo/CTDDG/REWORKED_CODE/saved_models/prose_mt_3x1024.sav

BLOCK 5.1 COMPLETE


In [20]:

# ============================================================
# BLOCK 5.2 — EXISTING EMBEDDING INSPECTION
# ============================================================

import numpy as np

EMBEDDING_CANDIDATES = {
    "Old embeddings 103":
        PROJECT_ROOT / "Results/embeddings_103",

    "Old embeddings 104":
        REWORKED_ROOT / "testing/Validity_Analysis/Embeddings_104/embeddings_104",

    "Project protein embeddings":
        REWORKED_ROOT / "PREPROCESSING_OUTPUT/bindingdb/protein_embeddings",
}

print("=" * 65)
print("EXISTING EMBEDDING FILES")
print("=" * 65)

for label, folder in EMBEDDING_CANDIDATES.items():

    print(f"\n{label}")
    print("Directory:", folder)

    if not folder.is_dir():
        print("Status: NOT FOUND")
        continue

    txt_files = sorted(
        folder.glob("protein*.txt"),
        key=lambda p: (
            int(p.stem[7:])
            if p.stem[7:].isdigit() else 999999
        )
    )

    h5_files = sorted(
        list(folder.glob("*.h5"))
        + list(folder.glob("*.hdf5"))
    )

    print("Protein TXT files:", len(txt_files))
    print("HDF5 files:", len(h5_files))

    vectors = []

    for path in txt_files[:5]:
        try:
            vector = np.loadtxt(path, dtype=np.float32).ravel()
            vectors.append(vector)

            print(
                f"  {path.name}: "
                f"dimension={vector.size}, "
                f"finite={bool(np.isfinite(vector).all())}"
            )
        except Exception as exc:
            print(f"  {path.name}: ERROR {exc}")

    if len(vectors) >= 2:
        same_shape = vectors[0].shape == vectors[1].shape
        identical = (
            np.array_equal(vectors[0], vectors[1])
            if same_shape else False
        )

        print(
            "First two embeddings exactly identical:",
            identical
        )

    for path in h5_files[:5]:
        print("  HDF5 file:", path.name)

print("\nBLOCK 5.2 COMPLETE")


EXISTING EMBEDDING FILES

Old embeddings 103
Directory: /home/muhamed/repo/CTDDG/Results/embeddings_103
Protein TXT files: 103
HDF5 files: 0
  protein1.txt: dimension=1024, finite=True
  protein2.txt: dimension=1024, finite=True
  protein3.txt: dimension=1024, finite=True
  protein4.txt: dimension=1024, finite=True
  protein5.txt: dimension=1024, finite=True
First two embeddings exactly identical: True

Old embeddings 104
Directory: /home/muhamed/repo/CTDDG/REWORKED_CODE/testing/Validity_Analysis/Embeddings_104/embeddings_104
Protein TXT files: 104
HDF5 files: 0
  protein1.txt: dimension=1024, finite=True
  protein2.txt: dimension=1024, finite=True
  protein3.txt: dimension=1024, finite=True
  protein4.txt: dimension=1024, finite=True
  protein5.txt: dimension=1024, finite=True
First two embeddings exactly identical: True

Project protein embeddings
Directory: /home/muhamed/repo/CTDDG/REWORKED_CODE/PREPROCESSING_OUTPUT/bindingdb/protein_embeddings
Protein TXT files: 0
HDF5 files: 0

BL

In [21]:

# ============================================================
# BLOCK 5.3 — FIND EMBEDDING CONFIGURATION IN CODE
# ============================================================

import os
import re
import json
from pathlib import Path

print("=" * 65)
print("SEARCHING MODEL CODE FOR EMBEDDING CONFIGURATION")
print("=" * 65)

PATTERN = re.compile(
    r"prose|bepler|prottrans|protein_embedding|"
    r"protein_emb|embedding_dim|embedding_size|"
    r"protein_feature|6165",
    re.IGNORECASE
)

SKIP_DIRS = {
    ".git", ".venv", "venv", "__pycache__",
    "node_modules", ".ipynb_checkpoints",
    "site-packages", "DATA", "PREPROCESSING_OUTPUT",
    "EVALUATION_OUTPUT"
}

SOURCE_FILES = []

for current_dir, dirs, filenames in os.walk(REWORKED_ROOT):

    dirs[:] = [
        d for d in dirs if d not in SKIP_DIRS
    ]

    for filename in filenames:

        path = Path(current_dir) / filename

        if path.suffix == ".py":
            SOURCE_FILES.append(path)

        elif (
            path.suffix == ".ipynb"
            and any(term in filename.lower() for term in [
                "fine", "tuning", "embed",
                "protein", "generation", "sample"
            ])
        ):
            SOURCE_FILES.append(path)

print("Candidate code files:", len(SOURCE_FILES))

matches = []

for path in SOURCE_FILES:

    try:
        # Skip unusually large notebooks.
        if path.stat().st_size > 15 * 1024**2:
            continue

        if path.suffix == ".ipynb":
            notebook = json.loads(path.read_text())
            lines = []

            for cell in notebook.get("cells", []):
                if cell.get("cell_type") == "code":
                    source = cell.get("source", [])
                    content = "".join(source) if isinstance(source, list) else source
                    lines.extend(content.splitlines())
        else:
            lines = path.read_text(
                encoding="utf-8", errors="replace"
            ).splitlines()

        for line_number, line in enumerate(lines, 1):

            if PATTERN.search(line):
                matches.append({
                    "file": str(path.relative_to(REWORKED_ROOT)),
                    "line": line_number,
                    "code": line.strip()[:160],
                })

    except Exception as exc:
        print("Could not inspect:", path.name, exc)

print("\nMatching code references:", len(matches))

for item in matches[:35]:
    print(f"\n{item['file']} | line {item['line']}")
    print(" ", item["code"])

if len(matches) > 35:
    print(f"\n... {len(matches) - 35} additional matches")

if not matches:
    print(
        "\nNo explicit embedding configuration found. "
        "We will inspect the fine-tuned checkpoint next."
    )

print("\nBLOCK 5.3 COMPLETE")


SEARCHING MODEL CODE FOR EMBEDDING CONFIGURATION
Candidate code files: 10
Could not inspect: Embeddings.ipynb Expecting value: line 1 column 1 (char 0)

Matching code references: 31

testing/embeddings.ipynb | line 2
  from bio_embeddings.embed import ProtTransBertBFDEmbedder

testing/embeddings.ipynb | line 11
  embedder = ProtTransBertBFDEmbedder()

testing/CTDDG_Five_Model_Generation.ipynb | line 57
  'dense_relu': False, 'n_c': 6165,

testing/CTDDG_Five_Model_Generation.ipynb | line 437
  protein_features=[projection(c) for projection in self.protein_projections]

testing/CTDDG_Five_Model_Generation.ipynb | line 438
  for conv,bn,pfeat in zip(self.convs,self.pre_bns,protein_features):

testing/CTDDG_Five_Model_Generation.ipynb | line 450
  def __init__(self,model_config,n_c=6165):

testing/CTDDG_Five_Model_Generation.ipynb | line 847
  required.update(conditioned_on_protein=True,protein_embedding_sha256=embedding_hash)

testing/CTDDG_Five_Model_Generation.ipynb | line 886
  or meta

In [22]:

# ============================================================
# BLOCK 5.4A — DISCOVER EXISTING PROSE EMBEDDINGS
# ============================================================

from pathlib import Path

print("=" * 65)
print("PROSE EMBEDDING FILE DISCOVERY")
print("=" * 65)

SEARCH_ROOTS = [
    REWORKED_ROOT / "PREPROCESSING_OUTPUT/bindingdb",
    REWORKED_ROOT / "finetuning",
    PROJECT_ROOT / "Results",
]

EXTENSIONS = {".h5", ".hdf5", ".npy", ".npz"}

for root in SEARCH_ROOTS:
    print(f"\nSearching: {root}")

    if not root.is_dir():
        print("Directory not found")
        continue

    files = sorted(
        p for p in root.rglob("*")
        if p.is_file()
        and p.suffix.lower() in EXTENSIONS
        and (
            "embed" in p.name.lower()
            or "prose" in p.name.lower()
            or "bepler" in p.name.lower()
        )
    )

    print("Candidate files found:", len(files))

    for path in files[:15]:
        print(" ", path)

    if len(files) > 15:
        print(f" ... and {len(files) - 15} more")

print("\nDiscovery complete.")


PROSE EMBEDDING FILE DISCOVERY

Searching: /home/muhamed/repo/CTDDG/REWORKED_CODE/PREPROCESSING_OUTPUT/bindingdb
Candidate files found: 0

Searching: /home/muhamed/repo/CTDDG/REWORKED_CODE/finetuning
Candidate files found: 0

Searching: /home/muhamed/repo/CTDDG/Results
Candidate files found: 0

Discovery complete.


In [23]:

# ============================================================
# BLOCK 5.4B — INSPECT FINETUNING EMBEDDING CONTRACT
# ============================================================

import json

NOTEBOOK_PATH = (
    REWORKED_ROOT
    / "finetuning/CTDDG_BindingDB_Finetuning.ipynb"
)

print("=" * 65)
print("FINETUNING EMBEDDING IMPLEMENTATION")
print("=" * 65)

assert NOTEBOOK_PATH.is_file(), NOTEBOOK_PATH

with NOTEBOOK_PATH.open("r", encoding="utf-8") as f:
    notebook = json.load(f)

KEYWORDS = (
    "load_prose_matrix",
    "unique_embeddings",
    "bepler",
    "h5py.File",
    "N_C",
    "C_train",
    "C_test",
    "prose_mt",
)

for cell_number, cell in enumerate(notebook["cells"], 1):

    if cell.get("cell_type") != "code":
        continue

    source = cell.get("source", [])
    code = "".join(source) if isinstance(source, list) else source
    lines = code.splitlines()

    matching = [
        i for i, line in enumerate(lines)
        if any(k.lower() in line.lower() for k in KEYWORDS)
    ]

    if not matching:
        continue

    print(f"\n{'=' * 60}")
    print(f"NOTEBOOK CELL {cell_number}")
    print("=" * 60)

    shown = set()

    for index in matching[:8]:
        for j in range(
            max(0, index - 3),
            min(len(lines), index + 7)
        ):
            shown.add(j)

    for j in sorted(shown):
        print(f"{j + 1:4d} | {lines[j][:180]}")

print("\nInspection complete. No models loaded.")


FINETUNING EMBEDDING IMPLEMENTATION

NOTEBOOK CELL 3
  19 | }
  20 | assert VARIANT in PRETRAIN_RUNS
  21 | PRETRAIN_RUN_DIR=PRETRAIN_RUNS[VARIANT]
  22 | PRETRAIN_CHECKPOINT=PRETRAIN_RUN_DIR/'checkpoint.pt'
  23 | PRETRAIN_META=PRETRAIN_RUN_DIR/'run_meta.json'
  24 | ATOM_TYPES_FILE=REPO_ROOT/'REWORKED_CODE/chembl_preprocessed/atom_types.txt'
  25 | RUN_DIR = REPO_ROOT / (
  26 |     'REWORKED_CODE/finetuning/finetuning_runs/'
  27 |     'ctddg_eq8_d1_leaked_full_bs32_v1'
  28 | )
  63 | 
  64 | MAX_ITER_PER_MOL=120
  65 | RECEPTIVE_FIELD=2
  66 | N_C=6165
  67 | 
  68 | random.seed(SEED);np.random.seed(SEED);torch.manual_seed(SEED)
  69 | if torch.cuda.is_available():torch.cuda.manual_seed_all(SEED)
  70 | assert OBJECTIVE in {'ctddg_eq8','cdgcn_nll'}
  71 | assert BATCH_SIZE>=2 and K_PATHS==5 and abs(ALPHA-0.8)<1e-9
  72 | assert 0 <= TOX_LAMBDA and MAX_EPOCHS>0 and PATIENCE>0
  74 | assert MONITOR_TOX_ACTION_DROP>0 and MONITOR_NONTOX_NLL_FACTOR>1
  75 | assert MONITOR_CONSECUTIVE_B

In [24]:

# ============================================================
# BLOCK 5.5A — PROSE SCRIPT PREFLIGHT
# ============================================================

import sys
import subprocess
from pathlib import Path

PROSE_ROOT = PROJECT_ROOT / "prose"
PROSE_SCRIPT = PROSE_ROOT / "embed_sequences.py"
PROSE_WEIGHTS = (
    PROSE_ROOT / "saved_models/prose_mt_3x1024.sav"
)

assert PROSE_SCRIPT.is_file(), PROSE_SCRIPT
assert PROSE_WEIGHTS.is_file(), PROSE_WEIGHTS

print("Python:", sys.executable)
print("ProSE script:", PROSE_SCRIPT)
print("Model weights:", PROSE_WEIGHTS)

result = subprocess.run(
    [sys.executable, str(PROSE_SCRIPT), "--help"],
    cwd=str(PROSE_ROOT),
    capture_output=True,
    text=True
)

print("\nHelp command exit code:", result.returncode)

if result.returncode != 0:
    print(result.stderr[-3000:])
    raise RuntimeError("ProSE script preflight failed.")

assert "--pool" in result.stdout
assert "--device" in result.stdout

print("\nPASS: ProSE script is accessible.")


Python: /home/muhamed/.conda/envs/ctddg_torch/bin/python
ProSE script: /home/muhamed/repo/CTDDG/prose/embed_sequences.py
Model weights: /home/muhamed/repo/CTDDG/prose/saved_models/prose_mt_3x1024.sav

Help command exit code: 0

PASS: ProSE script is accessible.


In [25]:

# ============================================================
# BLOCK 5.5B — PREPARE ONE VERIFIED PROTEIN
# ============================================================

import hashlib
import pandas as pd

SMOKE_FOLD = "1002_103"
SMOKE_PROTEIN_ID = "protein1"

SOURCE_FASTA = (
    EMBEDDING_OUTPUT_DIRS[SMOKE_FOLD]
    / "sequences"
    / f"{SMOKE_PROTEIN_ID}.fasta"
)

SMOKE_DIR = (
    EMBEDDING_OUTPUT_DIRS[SMOKE_FOLD]
    / "_smoke_test"
)

SMOKE_DIR.mkdir(parents=True, exist_ok=True)

SMOKE_INPUT = SMOKE_DIR / "p1.fasta"
SMOKE_OUTPUT = SMOKE_DIR / "prose_p1_avg.h5"

sequence = read_single_fasta(SOURCE_FASTA)

manifest = pd.read_csv(
    EMBEDDING_OUTPUT_DIRS[SMOKE_FOLD]
    / "protein_manifest.csv"
)

record = manifest.loc[
    manifest["protein_id"] == SMOKE_PROTEIN_ID
].iloc[0]

actual_hash = hashlib.sha256(
    sequence.encode("utf-8")
).hexdigest()

assert actual_hash == record["sequence_sha256"]
assert len(sequence) == int(record["sequence_length"])

fasta_content = (
    ">p1\n"
    + "\n".join(
        sequence[i:i + 80]
        for i in range(0, len(sequence), 80)
    )
    + "\n"
)

if SMOKE_INPUT.exists():
    assert SMOKE_INPUT.read_text() == fasta_content, (
        "Existing smoke-test input differs."
    )
else:
    SMOKE_INPUT.write_text(fasta_content)

print("=" * 60)
print("SMOKE-TEST PROTEIN READY")
print("=" * 60)
print("Fold:", SMOKE_FOLD)
print("Protein:", SMOKE_PROTEIN_ID)
print("Length:", len(sequence))
print("Sequence SHA256:", actual_hash[:20])
print("Input FASTA:", SMOKE_INPUT)
print("Output HDF5:", SMOKE_OUTPUT)


SMOKE-TEST PROTEIN READY
Fold: 1002_103
Protein: protein1
Length: 1390
Sequence SHA256: cc24e8701a5717fd1fa4
Input FASTA: /home/muhamed/repo/CTDDG/REWORKED_CODE/EVALUATION_OUTPUT/embeddings/1002_103/_smoke_test/p1.fasta
Output HDF5: /home/muhamed/repo/CTDDG/REWORKED_CODE/EVALUATION_OUTPUT/embeddings/1002_103/_smoke_test/prose_p1_avg.h5


In [26]:

# ============================================================
# BLOCK 5.5C — RUN PROSE ON ONE PROTEIN
# ============================================================

import os
import subprocess
import h5py
import numpy as np

assert torch.cuda.is_available(), (
    "GPU is unavailable. Run this in your GPU allocation."
)

TEMP_OUTPUT = (
    SMOKE_DIR / "prose_p1_avg.incomplete.h5"
)

if SMOKE_OUTPUT.exists():

    print("Previous smoke-test output exists.")
    print("Skipping generation; validating it in Block 5.5D.")

else:

    if TEMP_OUTPUT.exists():
        raise FileExistsError(
            f"Incomplete output already exists: {TEMP_OUTPUT}"
        )

    command = [
        sys.executable,
        str(PROSE_SCRIPT),
        "--model", "prose_mt",
        "--pool", "avg",
        "--device", "0",
        "--output", str(TEMP_OUTPUT),
        str(SMOKE_INPUT),
    ]

    print("Running ProSE on GPU 0...")
    print("Input:", SMOKE_INPUT)

    result = subprocess.run(
        command,
        cwd=str(PROSE_ROOT),
        capture_output=True,
        text=True
    )

    print("\nProSE log:")
    print((result.stdout + "\n" + result.stderr)[-5000:])

    if result.returncode != 0:
        raise RuntimeError(
            f"ProSE failed with exit code {result.returncode}"
        )

    # Validate before publishing the output file.
    with h5py.File(TEMP_OUTPUT, "r") as h5:

        assert list(h5.keys()) == ["p1"], (
            f"Unexpected HDF5 keys: {list(h5.keys())}"
        )

        vector = np.asarray(h5["p1"])

        assert vector.shape == (6165,), vector.shape
        assert np.isfinite(vector).all()
        assert np.linalg.norm(vector) > 0

    os.replace(TEMP_OUTPUT, SMOKE_OUTPUT)

    print("\nPASS: ProSE embedding generated successfully.")
    print("Saved to:", SMOKE_OUTPUT)


Running ProSE on GPU 0...
Input: /home/muhamed/repo/CTDDG/REWORKED_CODE/EVALUATION_OUTPUT/embeddings/1002_103/_smoke_test/p1.fasta

ProSE log:

# loading the pre-trained ProSE MT model
/home/muhamed/repo/CTDDG/prose/prose/models/multitask.py:37: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have 

In [27]:

# ============================================================
# BLOCK 5.5D — VERIFY GENERATED EMBEDDING
# ============================================================

import h5py
import numpy as np
import hashlib

print("=" * 60)
print("PROSE EMBEDDING VERIFICATION")
print("=" * 60)

assert SMOKE_OUTPUT.is_file(), (
    "Embedding output does not exist."
)

with h5py.File(SMOKE_OUTPUT, "r") as h5:

    print("HDF5 keys:", list(h5.keys()))

    assert set(h5.keys()) == {"p1"}

    vector = np.asarray(h5["p1"], dtype=np.float32)

print("Embedding dimensions:", vector.shape)
print("All values finite:", bool(np.isfinite(vector).all()))
print("Vector norm:", float(np.linalg.norm(vector)))
print("Minimum value:", float(vector.min()))
print("Maximum value:", float(vector.max()))
print("First 10 values:", vector[:10].round(5).tolist())

vector_hash = hashlib.sha256(
    vector.tobytes()
).hexdigest()

print("Embedding SHA256:", vector_hash[:20])

assert vector.shape == (6165,), (
    f"Wrong embedding size: {vector.shape}"
)

assert np.isfinite(vector).all(), (
    "Embedding contains NaN or infinity."
)

assert np.linalg.norm(vector) > 0, (
    "Embedding is a zero vector."
)

print("\n" + "=" * 60)
print("BLOCK 5.5 PASSED")
print("One verified protein embedding generated.")
print("Ready for full-dataset embedding generation.")
print("=" * 60)


PROSE EMBEDDING VERIFICATION
HDF5 keys: ['p1']
Embedding dimensions: (6165,)
All values finite: True
Vector norm: 3.77250599861145
Minimum value: -0.4353376030921936
Maximum value: 0.551520586013794
First 10 values: [0.04603999853134155, 0.04029000177979469, 0.05754999816417694, 0.04676000028848648, 0.029500000178813934, 0.03596999868750572, 0.05180000141263008, 0.05754999816417694, 0.03308999910950661, 0.059709999710321426]
Embedding SHA256: 33b0df91ea6fd538699d

BLOCK 5.5 PASSED
One verified protein embedding generated.
Ready for full-dataset embedding generation.


## Generate All ProSE Embeddings

In [28]:

# ============================================================
# BLOCK 6.1 — FULL EMBEDDING GENERATION CONFIGURATION
# ============================================================

from pathlib import Path
import sys
import torch

# ------------------------------------------------------------
# EDITABLE SETTINGS
# ------------------------------------------------------------

FOLDS_TO_GENERATE = [
    "1042_104",  # Shorter proteins; run first
    "1002_103",
]

PROSE_MODEL_NAME = "prose_mt"
PROSE_POOLING = "avg"
PROSE_DEVICE = "0"  # First visible GPU
EXPECTED_EMBEDDING_DIM = 6165

# ------------------------------------------------------------
# FILE LOCATIONS
# ------------------------------------------------------------

PROSE_ROOT = PROJECT_ROOT / "prose"
PROSE_SCRIPT = PROSE_ROOT / "embed_sequences.py"

PROSE_WEIGHTS = (
    PROSE_ROOT / "saved_models/prose_mt_3x1024.sav"
)

assert PROSE_SCRIPT.is_file(), PROSE_SCRIPT
assert PROSE_WEIGHTS.is_file(), PROSE_WEIGHTS
assert torch.cuda.is_available(), "A GPU allocation is required."

EMBEDDING_JOBS = {}

for fold in FOLDS_TO_GENERATE:

    output_dir = EMBEDDING_OUTPUT_DIRS[fold]

    EMBEDDING_JOBS[fold] = {
        "manifest": output_dir / "protein_manifest.csv",
        "input_fasta": output_dir / "prose_input.fasta",
        "protein_list": output_dir / "protein_list.txt",
        "output_h5": output_dir / "prose_embeddings_avg_6165.h5",
        "metadata": output_dir / "prose_run_metadata.json",
    }

print("=" * 65)
print("FULL PROSE EMBEDDING GENERATION")
print("=" * 65)

print("GPU:", torch.cuda.get_device_name(0))
print("Model:", PROSE_MODEL_NAME)
print("Pooling:", PROSE_POOLING)
print("Expected dimensions:", EXPECTED_EMBEDDING_DIM)

for fold in FOLDS_TO_GENERATE:
    print(f"\n{fold}")
    print("Output:", EMBEDDING_JOBS[fold]["output_h5"])

print("\nBLOCK 6.1 PASSED")


FULL PROSE EMBEDDING GENERATION
GPU: NVIDIA L40
Model: prose_mt
Pooling: avg
Expected dimensions: 6165

1042_104
Output: /home/muhamed/repo/CTDDG/REWORKED_CODE/EVALUATION_OUTPUT/embeddings/1042_104/prose_embeddings_avg_6165.h5

1002_103
Output: /home/muhamed/repo/CTDDG/REWORKED_CODE/EVALUATION_OUTPUT/embeddings/1002_103/prose_embeddings_avg_6165.h5

BLOCK 6.1 PASSED


In [29]:

# ============================================================
# BLOCK 6.2 — PREPARE ORDERED PROSE INPUT FILES
# ============================================================

import os
import tempfile
import pandas as pd

def write_if_missing_or_identical(path, content):

    path = Path(path)
    data = content.encode("utf-8")

    if path.exists():
        if path.read_bytes() != data:
            raise FileExistsError(
                f"Existing file has different contents: {path}"
            )
        return "REUSED"

    path.parent.mkdir(parents=True, exist_ok=True)

    temp_path = None

    try:
        with tempfile.NamedTemporaryFile(
            mode="wb",
            dir=path.parent,
            prefix=".prose_input_",
            delete=False
        ) as f:

            f.write(data)
            temp_path = Path(f.name)

        os.replace(temp_path, path)

    finally:
        if temp_path is not None:
            temp_path.unlink(missing_ok=True)

    return "CREATED"


for fold in FOLDS_TO_GENERATE:

    job = EMBEDDING_JOBS[fold]
    manifest = pd.read_csv(job["manifest"])

    assert len(manifest) == len(FOLD_PROTEINS[fold])

    fasta_records = []
    protein_list = []

    for index, row in enumerate(
        manifest.itertuples(index=False), start=1
    ):

        assert row.protein_id == f"protein{index}"

        sequence = read_single_fasta(
            Path(row.fasta_path)
        )

        assert sequence_sha256(sequence) == row.sequence_sha256
        assert len(sequence) == row.sequence_length

        fasta_records.append(
            f">p{index}\n"
            + "\n".join(
                sequence[i:i + 80]
                for i in range(0, len(sequence), 80)
            )
            + "\n"
        )

        protein_list.append(sequence)

    fasta_content = "".join(fasta_records)
    list_content = "\n".join(protein_list) + "\n"

    fasta_status = write_if_missing_or_identical(
        job["input_fasta"], fasta_content
    )

    list_status = write_if_missing_or_identical(
        job["protein_list"], list_content
    )

    print(f"\nFOLD: {fold}")
    print("Proteins:", len(protein_list))
    print("FASTA:", fasta_status)
    print("Protein list:", list_status)
    print("First sequence length:", len(protein_list[0]))

print("\nBLOCK 6.2 PASSED")
print("All ProSE input sequences verified.")



FOLD: 1042_104
Proteins: 99
FASTA: CREATED
Protein list: CREATED
First sequence length: 683

FOLD: 1002_103
Proteins: 103
FASTA: CREATED
Protein list: CREATED
First sequence length: 1390

BLOCK 6.2 PASSED
All ProSE input sequences verified.


In [30]:

# ============================================================
# BLOCK 6.3 — GENERATE FULL PROSE EMBEDDINGS
# ============================================================

import subprocess
import os
import h5py
import numpy as np

for fold in FOLDS_TO_GENERATE:

    job = EMBEDDING_JOBS[fold]

    output_path = job["output_h5"]

    temp_path = output_path.with_name(
        "prose_embeddings_avg_6165.incomplete.h5"
    )

    print("\n" + "=" * 65, flush=True)
    print(f"GENERATING EMBEDDINGS: {fold}", flush=True)
    print("=" * 65, flush=True)

    if output_path.exists():
        print(
            "Output already exists; skipping generation.",
            flush=True
        )
        print("It will be checked in Block 6.4.")
        continue

    if temp_path.exists():
        raise FileExistsError(
            f"Incomplete file from an earlier run: {temp_path}\n"
            "Inspect or remove that incomplete file before retrying."
        )

    command = [
        sys.executable,
        str(PROSE_SCRIPT),
        "--model", PROSE_MODEL_NAME,
        "--pool", PROSE_POOLING,
        "--device", PROSE_DEVICE,
        "--output", str(temp_path),
        str(job["input_fasta"]),
    ]

    print("Input:", job["input_fasta"], flush=True)
    print("Expected proteins:", len(FOLD_PROTEINS[fold]))
    print("Running ProSE...", flush=True)

    process = subprocess.Popen(
        command,
        cwd=str(PROSE_ROOT),
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1
    )

    for line in process.stdout:
        print(line.rstrip(), flush=True)

    return_code = process.wait()

    if return_code != 0:
        raise RuntimeError(
            f"ProSE failed for {fold}, "
            f"exit code {return_code}. "
            f"Inspect: {temp_path}"
        )

    # Validate the completed temporary HDF5 file.
    expected_count = len(FOLD_PROTEINS[fold])
    expected_keys = {
        f"p{i}" for i in range(1, expected_count + 1)
    }

    with h5py.File(temp_path, "r") as h5:

        assert set(h5.keys()) == expected_keys, (
            f"{fold}: HDF5 protein IDs do not match."
        )

        for index in range(1, expected_count + 1):

            vector = np.asarray(h5[f"p{index}"])

            assert vector.shape == (
                EXPECTED_EMBEDDING_DIM,
            ), (fold, index, vector.shape)

            assert np.isfinite(vector).all(), (
                fold, index, "nonfinite embedding"
            )

            assert np.linalg.norm(vector) > 0, (
                fold, index, "zero embedding"
            )

    # Publish only after the full fold passes.
    os.replace(temp_path, output_path)

    print(f"\nPASS: {fold}")
    print("Proteins embedded:", expected_count)
    print("Saved:", output_path)

print("\nBLOCK 6.3 COMPLETE")



GENERATING EMBEDDINGS: 1042_104
Input: /home/muhamed/repo/CTDDG/REWORKED_CODE/EVALUATION_OUTPUT/embeddings/1042_104/prose_input.fasta
Expected proteins: 99
Running ProSE...
# loading the pre-trained ProSE MT model
/home/muhamed/repo/CTDDG/prose/prose/models/multitask.py:37: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any 

In [31]:

# ============================================================
# BLOCK 6.4 — FINAL EMBEDDING QUALITY VERIFICATION
# ============================================================

import hashlib
import h5py
import numpy as np
import pandas as pd

EMBEDDING_VALIDATION = {}

for fold in FOLDS_TO_GENERATE:

    job = EMBEDDING_JOBS[fold]

    manifest = pd.read_csv(job["manifest"])
    expected_count = len(manifest)

    norms = []
    vector_hashes = []

    with h5py.File(job["output_h5"], "r") as h5:

        expected_keys = {
            f"p{i}" for i in range(1, expected_count + 1)
        }

        assert set(h5.keys()) == expected_keys

        for index in range(1, expected_count + 1):

            vector = np.asarray(
                h5[f"p{index}"],
                dtype=np.float32
            )

            assert vector.shape == (6165,)
            assert np.isfinite(vector).all()

            norm = float(np.linalg.norm(vector))
            assert norm > 0

            norms.append(norm)

            vector_hashes.append(
                hashlib.sha256(
                    vector.tobytes()
                ).hexdigest()
            )

        # Compare against our original single-protein test.
        if fold == "1002_103":

            smoke_path = (
                EMBEDDING_OUTPUT_DIRS[fold]
                / "_smoke_test/prose_p1_avg.h5"
            )

            with h5py.File(smoke_path, "r") as smoke:

                old_vector = np.asarray(
                    smoke["p1"], dtype=np.float32
                )

            new_vector = np.asarray(
                h5["p1"], dtype=np.float32
            )

            maximum_difference = float(
                np.max(
                    np.abs(old_vector - new_vector)
                )
            )

            print(
                "\nSmoke-test maximum absolute difference:",
                maximum_difference
            )

            assert np.allclose(
                old_vector,
                new_vector,
                rtol=1e-4,
                atol=1e-5
            ), (
                "Full-run protein1 does not reproduce "
                "the earlier smoke-test embedding."
            )

    duplicate_vectors = (
        len(vector_hashes) - len(set(vector_hashes))
    )

    assert duplicate_vectors == 0, (
        f"{fold}: {duplicate_vectors} duplicate embeddings"
    )

    EMBEDDING_VALIDATION[fold] = {
        "proteins": expected_count,
        "embedding_dimension": 6165,
        "duplicate_vectors": duplicate_vectors,
        "min_norm": round(min(norms), 5),
        "median_norm": round(float(np.median(norms)), 5),
        "max_norm": round(max(norms), 5),
    }

print("\n" + "=" * 65)
print("FINAL EMBEDDING VALIDATION")
print("=" * 65)

display(
    pd.DataFrame.from_dict(
        EMBEDDING_VALIDATION,
        orient="index"
    )
)

print("\nBLOCK 6.4 PASSED")
print("All embeddings have correct dimensions and finite values.")



Smoke-test maximum absolute difference: 0.0

FINAL EMBEDDING VALIDATION


,proteins,embedding_dimension,duplicate_vectors,min_norm,median_norm,max_norm
1042_104,99,6165,0,2.87914,4.31802,7.89257
1002_103,103,6165,0,3.13667,3.86598,7.89257



BLOCK 6.4 PASSED
All embeddings have correct dimensions and finite values.


In [32]:

# ============================================================
# BLOCK 6.5 — SAVE EMBEDDING PROVENANCE
# ============================================================

import json
import hashlib
import torch

def sha256_of_file(path):

    digest = hashlib.sha256()

    with Path(path).open("rb") as f:
        for chunk in iter(
            lambda: f.read(1024 * 1024),
            b""
        ):
            digest.update(chunk)

    return digest.hexdigest()


model_hash = sha256_of_file(PROSE_WEIGHTS)
script_hash = sha256_of_file(PROSE_SCRIPT)

for fold in FOLDS_TO_GENERATE:

    job = EMBEDDING_JOBS[fold]

    metadata = {
        "fold": fold,
        "encoder": "ProSE multitask",
        "model_name": PROSE_MODEL_NAME,
        "model_path": str(PROSE_WEIGHTS),
        "model_sha256": model_hash,
        "script_path": str(PROSE_SCRIPT),
        "script_sha256": script_hash,
        "pooling": PROSE_POOLING,
        "embedding_dimension": EXPECTED_EMBEDDING_DIM,
        "protein_count": len(FOLD_PROTEINS[fold]),
        "id_mapping": "cleaned_first_occurrence_v1",
        "input_fasta": str(job["input_fasta"]),
        "input_fasta_sha256": sha256_of_file(
            job["input_fasta"]
        ),
        "protein_manifest": str(job["manifest"]),
        "protein_manifest_sha256": sha256_of_file(
            job["manifest"]
        ),
        "output_h5": str(job["output_h5"]),
        "output_h5_sha256": sha256_of_file(
            job["output_h5"]
        ),
        "pytorch_version": torch.__version__,
    }

    metadata_text = json.dumps(
        metadata, indent=2
    ) + "\n"

    write_if_missing_or_identical(
        job["metadata"],
        metadata_text
    )

    print(
        f"PASS: {fold} — provenance saved"
    )

print("\nBLOCK 6 COMPLETE.")
print("Protein embeddings and metadata are saved.")


PASS: 1042_104 — provenance saved
PASS: 1002_103 — provenance saved

BLOCK 6 COMPLETE.
Protein embeddings and metadata are saved.


## Final Verification

In [34]:
# ============================================================
# BLOCK 7.1 — VERIFY SAVED EMBEDDING FILES
# ============================================================

import json
import hashlib
import h5py
import numpy as np
import pandas as pd
from pathlib import Path

FINAL_EMBEDDING_SUMMARY = {}

for fold in ["1002_103", "1042_104"]:

    folder = EMBEDDING_OUTPUT_DIRS[fold]

    manifest = pd.read_csv(folder / "protein_manifest.csv")

    metadata = json.loads(
        (folder / "prose_run_metadata.json").read_text()
    )

    h5_path = folder / "prose_embeddings_avg_6165.h5"

    assert metadata["embedding_dimension"] == 6165
    assert metadata["pooling"] == "avg"
    assert metadata["protein_count"] == len(manifest)

    # Verify output file has not changed since generation.
    assert (
        sha256_of_file(h5_path)
        == metadata["output_h5_sha256"]
    )

    with h5py.File(h5_path, "r") as h5:

        expected_keys = {
            f"p{i}" for i in range(1, len(manifest) + 1)
        }

        assert set(h5.keys()) == expected_keys

        for i, row in enumerate(
            manifest.itertuples(index=False), start=1
        ):

            assert row.protein_id == f"protein{i}"

            sequence = read_single_fasta(
                Path(row.fasta_path)
            )

            assert (
                sequence_sha256(sequence)
                == row.sequence_sha256
            )

            vector = np.asarray(h5[f"p{i}"])

            assert vector.shape == (6165,)
            assert np.isfinite(vector).all()

    FINAL_EMBEDDING_SUMMARY[fold] = {
        "proteins": len(manifest),
        "dimensions": 6165,
        "sequence_check": "PASS",
        "embedding_check": "PASS",
        "checksum_check": "PASS",
    }

print("=" * 65)
print("FINAL EMBEDDING FILE VERIFICATION")
print("=" * 65)

display(
    pd.DataFrame.from_dict(
        FINAL_EMBEDDING_SUMMARY,
        orient="index"
    )
)

print("\nBLOCK 7.1 PASSED")


FINAL EMBEDDING FILE VERIFICATION


,proteins,dimensions,sequence_check,embedding_check,checksum_check
1002_103,103,6165,PASS,PASS,PASS
1042_104,99,6165,PASS,PASS,PASS



BLOCK 7.1 PASSED


In [35]:

# ============================================================
# BLOCK 7.2 — VERIFY CROSS-FOLD CONSISTENCY
# ============================================================

folder_a = EMBEDDING_OUTPUT_DIRS["1002_103"]
folder_b = EMBEDDING_OUTPUT_DIRS["1042_104"]

manifest_a = pd.read_csv(
    folder_a / "protein_manifest.csv"
)

manifest_b = pd.read_csv(
    folder_b / "protein_manifest.csv"
)

shared = manifest_a.merge(
    manifest_b,
    on="sequence_sha256",
    suffixes=("_a", "_b")
)

print("Shared protein sequences:", len(shared))

assert len(shared) == 5

comparison_rows = []

with h5py.File(
    folder_a / "prose_embeddings_avg_6165.h5", "r"
) as h5_a, h5py.File(
    folder_b / "prose_embeddings_avg_6165.h5", "r"
) as h5_b:

    for row in shared.itertuples(index=False):

        id_a = f"p{row.cleaned_order_id_a}"
        id_b = f"p{row.cleaned_order_id_b}"

        vector_a = np.asarray(h5_a[id_a], dtype=np.float32)
        vector_b = np.asarray(h5_b[id_b], dtype=np.float32)

        maximum_difference = float(
            np.max(np.abs(vector_a - vector_b))
        )

        matched = bool(np.allclose(
            vector_a,
            vector_b,
            rtol=1e-4,
            atol=1e-5
        ))

        comparison_rows.append({
            "fold_a_id": row.protein_id_a,
            "fold_b_id": row.protein_id_b,
            "max_absolute_difference": maximum_difference,
            "matched": matched,
        })

comparison_df = pd.DataFrame(comparison_rows)

display(comparison_df)

assert comparison_df["matched"].all(), (
    "Identical sequences produced inconsistent embeddings."
)

print("\nBLOCK 7.2 PASSED")
print("All five shared proteins have matching embeddings.")


Shared protein sequences: 5


,fold_a_id,fold_b_id,max_absolute_difference,matched
0,protein99,protein13,0.0,True
1,protein100,protein16,0.0,True
2,protein101,protein38,0.0,True
3,protein102,protein49,0.0,True
4,protein103,protein89,0.0,True



BLOCK 7.2 PASSED
All five shared proteins have matching embeddings.


In [36]:

# ============================================================
# BLOCK 7.3 — SAVE FINAL VERIFICATION REPORT
# ============================================================

import json

report = {
    "status": "PASSED",
    "embedding_model": "ProSE multitask",
    "pooling": "avg",
    "embedding_dimension": 6165,
    "id_namespace": "cleaned_first_occurrence_v1",
    "folds": FINAL_EMBEDDING_SUMMARY,
    "total_fold_entries": sum(
        x["proteins"]
        for x in FINAL_EMBEDDING_SUMMARY.values()
    ),
    "shared_sequences_between_folds": len(comparison_df),
    "unique_sequences_across_folds": (
        sum(
            x["proteins"]
            for x in FINAL_EMBEDDING_SUMMARY.values()
        ) - len(comparison_df)
    ),
    "cross_fold_consistency": "PASSED",
    "legacy_fasta_mapping_verified": False,
    "note": (
        "Protein IDs are assigned by first occurrence "
        "in the cleaned dataset. They must not be assumed "
        "to match previous generated-sample protein IDs."
    ),
}

report_path = (
    EVALUATION_ROOT
    / "embeddings"
    / "embedding_validation_report.json"
)

write_if_missing_or_identical(
    report_path,
    json.dumps(report, indent=2) + "\n"
)

print("=" * 65)
print("NOTEBOOK 1 — FINAL RESULTS")
print("=" * 65)

print("Status:", report["status"])
print("Total protein entries:", report["total_fold_entries"])
print("Distinct protein sequences:", report["unique_sequences_across_folds"])
print("Embedding dimension:", report["embedding_dimension"])
print("Cross-fold check:", report["cross_fold_consistency"])
print("Verification report:", report_path)

print("\nNOTEBOOK 1 COMPLETE!")


NOTEBOOK 1 — FINAL RESULTS
Status: PASSED
Total protein entries: 202
Distinct protein sequences: 197
Embedding dimension: 6165
Cross-fold check: PASSED
Verification report: /home/muhamed/repo/CTDDG/REWORKED_CODE/EVALUATION_OUTPUT/embeddings/embedding_validation_report.json

NOTEBOOK 1 COMPLETE!
